# Results

In [1]:
import duckdb
import pandas as pd
import sys
import spacy
import os
sys.path.append('..')

from src.utils import make_corpus, preprocess_spacy
from src.semantic import build_semantic_index, semantic_search
from src.bm25 import bm25_search, build_bm25

## Build Corpus and Preprocessing

In [2]:
# Read data and drop missing values
c2 = duckdb.connect()
data = c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()
data.dropna(subset=['product_title'], inplace=True)

In [3]:
# Extract fields for retrieval
cols = ['product_title', 'main_category', 'store', 'title', 'text']

corpus = make_corpus(df=data, cols=cols, asin="asin")

In [4]:
# preprocess corpus and save it
os.makedirs('data/processed', exist_ok=True)

# if corpus is already processed and saved, pass to save time
corpus_path = "../data/processed/preprocessed_corpus.csv"
if os.path.exists(corpus_path):
    corpus = pd.read_csv(corpus_path)
else:
    nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])
    corpus["text"] = [preprocess_spacy(text) for text in nlp.pipe(corpus["text"])]
    corpus.to_csv(corpus_path)

## Save Indices for BM25 and Embeddings

In [5]:
# BM25 index

pickle_path = "../data/processed/bm25.pkl"
bm25 = build_bm25(pickle_path, corpus)

In [6]:
# Semantic index 
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
semantic_index_path = '../data/processed/embedding.faiss'

if not os.path.exists(semantic_index_path):
    build_semantic_index(corpus, model, semantic_index_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieve Results

In [7]:
# Set max width for results dataframe
pd.set_option('display.max_colwidth', 200)

In [8]:
queries = ["Wet wipes",
           "Bar Soap",
           "Small hair dryer",
           "The best air humidifer with essential oil",
           "mineral sunscreen for babies",
           "hair spray that last more than 6 hours",
           "Best Vitamins or supplements to take for pregnant women",
           "equipments for stretching at home",
           "sunrise lamp that will help me to wake up in the morning",
           "Something to relieve my back pain"]

In [9]:
for q in queries:
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    display(bm25_search(q, pickle_path, data))

    print("\nSemantic search top results:")
    display(semantic_search(q, semantic_index_path, model, data))

QUERY: Wet wipes

BM25 top results:


,product_title,text,rating,score
9003,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes),that the wipes are thick and not thin.,5.0,15.452317
6634,Lens Wipes Pre-moistened Eye Glasses Cleaner Wipes 120 Individually Packaged for Cleaning Glasses Sunglasses Computer Screens Touchscreens Monitors,"I bought these based on the reviews, but they are not wet enough. My optician said that the wetter the better when it comes to cleansing your glasses. In fact, he recommended a spray and cloth bet...",1.0,12.032342
12808,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)",These were really wipes more for use in medical not full sheets for glasses.,1.0,11.516642
6716,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)","What we liked most was that it does an excellent job of cleaning your eyeglasses, better than anything we have used, BUT it is a two part system. First you use the wet one, then the dry one. The ...",2.0,11.441593
15207,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)",Do not buy this item. The wipes are so small they won't cover the lens. It's a pain that you need 2 wipes to try to clean your glasses. Save your money and buy some other product.,1.0,10.531983



Semantic search top results:


,product_title,text,rating,score
460,"Pampers Baby Fresh Water Baby Wipes 3X Pop-Top Packs, 192 Count",Baby wipes sure have improved since I used them on my daughter who is now an adult. Many options are available now and there are noticeable differences among the offerings in the marketplace. I ...,5.0,0.920217
6634,Lens Wipes Pre-moistened Eye Glasses Cleaner Wipes 120 Individually Packaged for Cleaning Glasses Sunglasses Computer Screens Touchscreens Monitors,"I bought these based on the reviews, but they are not wet enough. My optician said that the wetter the better when it comes to cleansing your glasses. In fact, he recommended a spray and cloth bet...",1.0,0.876085
2377,Rinse Free Sponge Bath Wipes (30-pack) | Extra Large & Thick No Rinse Waterless Body Wipe Sponges - Disposable Shower Cleaning Wash Cloths for Kids Adults Surgery Elder Care Gym Travel & More (1 P...,"I was very pleased with these rinse-free bath cloths by Nurture. They are larger than I expected, thick and soft. With just a little bit of water and a couple squeezes, they produce a nice lather ...",4.0,0.851884
4990,"Wet-it Skrubba New European Scrubby Non-Scratching Scouring Pads (Set of 3, Stripe Chicken Paisley)","Love these scrubbers, colorful and work well, NOT scented",5.0,0.849512
230,Rinse Free Sponge Bath Wipes (30-pack) | Extra Large & Thick No Rinse Waterless Body Wipe Sponges - Disposable Shower Cleaning Wash Cloths for Kids Adults Surgery Elder Care Gym Travel & More (1 P...,When you can't get in the shower and want to feel refreshed or just clean these are really good to have around. Thick enough that you won't easily drive your fingers through the material but thin ...,4.0,0.840174


QUERY: Bar Soap

BM25 top results:


...

,product_title,text,rating,score
18743,"Verilux Original Natural Spectrum Deluxe Floor Lamp, Ivory","I was sitting on my bed last night doing some embroidery and I noticed that I really needed some more light on the ""subject."" My hubby reminded me that I had the Verilux lamp still in the box tha...",5.0,1.059190
2847,"Verilux Original Natural Spectrum Deluxe Floor Lamp, Ivory","I bought this as a gift for my mom, it was at the very least a bit of a pain to put together and cheaply made. It required electrical tape and a good bit of cursing! Is now working ok. The gril...",4.0,1.023793
16576,Indus Classic Pine Himalayan Salt Crystal Lamp Natural Ion Theray 2.25 Kg,"This was my first salt lamp. It is very well made, interesting to look at and a nice solid size. It lights up really well. The photo on amazon doesn't do it justice IMO and makes it look like t...",5.0,1.013871
2796,Naturebright L6060 Per2 Led Daylight Lamp,"This functions well as daylight alarm. It's best feature is that the LED's arc out over the bed and shine down, as opposed to other models which shine straight out horizontally, which means you ha...",3.0,0.727630
18956,Naturebright L6060 Per2 Led Daylight Lamp,"I tried the Phillips previously (see review there) but just couldn't get it to work. The NatureBright is easy to figure out, the controls make a lot more sense and you can easily see that it's se...",4.0,0.675205


QUERY: Something to relieve my back pain

BM25 top results:


,product_title,text,rating,score
491,"Neck Stretcher Cervical Neck Traction Device Over Door for Home Use,Portable Neck Traction for Neck Pain Relief, Physical Therapy AIDS for Neck Spine Decompressor (Black)",First time user. It took me a while a find out how to assemble them. After some adjustments I find a way to use it. And it does helps relieve my neck pain.,5.0,10.952530
12649,Shoulder Wrap Gel Ice Hot Cold Pack for Shoulder Injury Pain Relief Therapy Rotator Cuff Rheumatoid Arthritis Treatment Osteoarthritis Bursitis Tendinitis AC Joint Sports Injuries,"I use it alot, great product, helps relieve my muscle pain.",5.0,10.870432
14553,"ZSZBACE Posture Corrector Back Brace for Men and Women- Relieve Back Pain, Align Spain, Correct Kyphosis (XXL)",It help with my back pain,5.0,10.828765
671,Korean Red Ginseng Patch Powerstrip Energy Pain Relief - 20 Patches,These patches really do relieve the pain. They can be cut to whatever size you need.They stick extremely well.,5.0,10.687405
6944,"Real Time Pain Relief George Foreman's Knockout Formula, 1 Oz GoPak",This stuff does exactly what it says it will do. Relieve pain and make you feel better. Happy!,5.0,10.650396



Semantic search top results:


,product_title,text,rating,score
10646,"REEHUT Foam Roller - (6""x36"") Firm High-Density Muscle Rollers for Deep Tissue Muscle Massage Trigger Point for Pain Relief & Pilates with User E-Book (90cm)","Is what your back muscles and spine will say!<br />Be smart, use as advised. Made of dark grey/black Styrofoam.",5.0,0.847140
2599,"Lower Back Stretcher Spine Board-Back Stretcher for Lower Back Pain Relief, Back Cracking Device, Sciatica, Scoliosis, Spine Deck, on Bed,on Chair, Yoga Mat & Car-Spine Stretcher",This really gives a nice stretch. I use it on my office chair daily. Ever since I started using this the lower back pain I would have at the end of the day is gone.,5.0,0.810345
12300,"Genericb Back Massage Stretcher Arch Magic Message Stretcher Back Stretcher Lumbar Support Device, Lower and Upper Back Pain Relief Relax Mate Spine Pain Relief Chiropractic",A little to stiff for sore back muscles. need to use a soft pad with it,4.0,0.758031
2628,"Acupuncture Mat and Pillow Set-Relieve Your Stress, Back, Neck, and Sciatic Pain(99% Cotton Fabric, Plastic Spikes and Foam core) Green",Really helped with tension in my shoulders!,5.0,0.709054
4439,"Large Heating and Cooling Reusable Wrap by Soothing Company for Back Pain, Stomach Cramps, Sore Neck, Shoulders, Legs, Joint and Muscle Pain Hot and Cold Therapy to Support Healing Injuries",This absolutely what I have been looking for to use for my low back.,5.0,0.699777
